# 🧬 Reasoning-Intensive Retrieval — حل سارة
**الفكرة باختصار:**
1. **BM25**: بحث بالكلمات (أسرع وأقوى من العشوائي بكثير).
2. **نموذج لغوي (LLM)** يشرح كل سؤال علمياً ويطلع المصطلحات العلمية → نبحث بهذا الشرح (هنا الفايدة الكبيرة لأن الإجابة ما تشارك كلمات السؤال).
3. **بحث دلالي (Embeddings)**: يفهم المعنى مو الكلمات.
4. **الدمج**: نجمع درجات كل الطرق بأوزان نختارها تلقائياً على أسئلة التدريب (بعد ما نقصّرها لتشبه أسئلة الاختبار).
5. **إعادة ترتيب** أفضل 100 مقطع بنموذج أقوى (Reranker) — يُستخدم فقط إذا ثبت أنه يحسّن.

⚙️ **قبل التشغيل**: من القائمة الجانبية Settings → **Accelerator: GPU T4 x2** و **Internet: On**.

In [ ]:
# 1) التجهيز وتحميل البيانات
import os, glob, re, math, json, gc, time, itertools
import numpy as np, pandas as pd
import torch

hits = glob.glob('/kaggle/input/**/corpus.csv', recursive=True)
DATA = os.path.dirname(hits[0]) if hits else '.'
OUT = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
corpus  = pd.read_csv(f'{DATA}/corpus.csv')
train_q = pd.read_csv(f'{DATA}/train_queries.csv')
qrels   = pd.read_csv(f'{DATA}/train_qrels.csv')
test_q  = pd.read_csv(f'{DATA}/test_queries.csv')
sample  = pd.read_csv(f'{DATA}/sample_submission.csv')
corpus['text'] = corpus['text'].fillna('').astype(str)
doc_ids = corpus.doc_id.values
N = len(corpus)
gold = qrels.groupby('query_id').doc_id.apply(set).to_dict()
train_q = train_q[train_q.query_id.isin(gold)].reset_index(drop=True)

# أسئلة التدريب طويلة والاختبار قصيرة → نقصّر أسئلة التدريب لنفس طول الاختبار
TEST_LEN = int(test_q['query'].str.split().str.len().median())
train_q['short'] = train_q['query'].apply(lambda q: ' '.join(str(q).split()[:max(TEST_LEN, 8)]))

HAS_GPU = torch.cuda.is_available()
print('GPUs:', torch.cuda.device_count(), '| test median words:', TEST_LEN)
print(train_q[['query', 'short']].head(3).to_string())

In [ ]:
# 2) أدوات التقييم (nDCG@10) وأخذ أفضل k
def ndcg10(ranked, g, k=10):
    dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(ranked[:k]) if d in g)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(g), k)))
    return dcg / idcg if idcg else 0.0

def topk(S, k):
    part = np.argpartition(-S, k, axis=1)[:, :k]
    order = np.argsort(-np.take_along_axis(S, part, 1), axis=1)
    return np.take_along_axis(part, order, 1)

TRAIN_IDS = train_q.query_id.tolist()
def score(S, qids=TRAIN_IDS):
    T = topk(S, 10)
    return 100 * np.mean([ndcg10(list(doc_ids[t]), gold[q]) for t, q in zip(T, qids)])

def norm(S, k=100):
    """نوحّد مقياس الدرجات: أفضل مقطع = 1 والمقطع رقم 100 = 0"""
    T = topk(S, k)
    top1 = np.take_along_axis(S, T[:, :1], 1)
    topK = np.take_along_axis(S, T[:, -1:], 1)
    return np.clip((S - topK) / (top1 - topK + 1e-9), -0.5, None).astype(np.float32)

In [ ]:
# 3) BM25 (بحث بالكلمات) — مكتوب يدوياً عشان ما نحتاج أي تثبيت
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
try:
    from nltk.stem import PorterStemmer
    _ps, _cache = PorterStemmer(), {}
    def stem(w):
        s = _cache.get(w)
        if s is None: s = _cache[w] = _ps.stem(w)
        return s
except Exception:
    stem = lambda w: w
STOP = set(ENGLISH_STOP_WORDS)
def analyzer(text):
    return [stem(w) for w in re.findall(r'[a-z0-9]+', str(text).lower()) if w not in STOP and len(w) > 1]

class BM25:
    def __init__(self, texts, k1=0.9, b=0.4):
        self.vec = CountVectorizer(analyzer=analyzer)
        X = self.vec.fit_transform(texts).tocsr().astype(np.float32)
        n = X.shape[0]
        df = np.bincount(X.indices, minlength=X.shape[1])
        idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        dl = np.asarray(X.sum(1)).ravel()
        rows = np.repeat(np.arange(n), np.diff(X.indptr))
        tf = X.data
        X.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[rows] / dl.mean())) * idf[X.indices]
        self.WT = X.T.tocsr()
    def scores(self, queries):
        Q = self.vec.transform(list(queries))
        return np.asarray((Q @ self.WT).todense(), dtype=np.float32)

t0 = time.time()
bm25 = BM25(corpus.text.tolist())
print(f'BM25 built in {time.time()-t0:.0f}s')
print('BM25 train (full questions) :', round(score(bm25.scores(train_q['query'])), 2))
print('BM25 train (short questions):', round(score(bm25.scores(train_q['short'])), 2))

In [ ]:
# 4) هل المقاطع المتجاورة في الملف من نفس المقالة؟ (لو نعم نستفيد منها)
pos = {d: i for i, d in enumerate(doc_ids)}
hit = tot = 0
for q, g in gold.items():
    ii = {pos[d] for d in g if d in pos}
    for i in ii:
        tot += 1; hit += ((i - 1) in ii) or ((i + 1) in ii)
print(f'Gold passages whose neighbour is also gold: {100*hit/tot:.1f}%  (random ≈ 0%)')

def smooth(S, beta):
    if beta == 0: return S
    left = np.pad(S[:, :-1], ((0, 0), (1, 0)))
    right = np.pad(S[:, 1:], ((0, 0), (0, 1)))
    return S + beta * np.maximum(left, right)

## 5) النموذج اللغوي يشرح كل سؤال (أهم خطوة)
ياخذ تقريباً 10 دقايق. لو فشل أول نموذج يجرب الثاني تلقائياً.

In [ ]:
USE_LLM = HAS_GPU
LLM_CANDIDATES = ['Qwen/Qwen3-4B-Instruct-2507', 'Qwen/Qwen2.5-3B-Instruct']
SYS = 'You are an expert biologist helping a search engine find reference passages.'
PROMPT = """A user asked this biology question:

{q}

Do not answer in one line. Instead:
1. State the core scientific question.
2. List the key biological concepts, mechanisms, processes, structures, organisms and technical terms (with synonyms and likely Wikipedia article titles) that a relevant reference passage would discuss.
3. Write a detailed explanation (about 200 words) in the style of a textbook or Wikipedia article that answers the question using those terms."""

cache_path = f'{OUT}/expansions.json'
EXP = json.load(open(cache_path)) if os.path.exists(cache_path) else {}
todo = [t for t in list(train_q['short']) + list(test_q['query']) if t not in EXP]

def run_llm(name, texts, bs=8, max_new=360):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(name, padding_side='left')
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map={'': 0}).eval()
    outs = {}
    for i in range(0, len(texts), bs):
        batch = texts[i:i + bs]
        prompts = [tok.apply_chat_template([{'role': 'system', 'content': SYS},
                                            {'role': 'user', 'content': PROMPT.format(q=t[:2000])}],
                                           tokenize=False, add_generation_prompt=True) for t in batch]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda:0')
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                               repetition_penalty=1.05, pad_token_id=tok.pad_token_id)
        for t, o in zip(batch, tok.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)):
            outs[t] = re.sub(r'<think>.*?</think>', '', o, flags=re.S).strip()
        print(f'{min(i+bs, len(texts))}/{len(texts)}', end=' | ')
    del model; gc.collect(); torch.cuda.empty_cache()
    return outs

if USE_LLM and todo:
    for name in LLM_CANDIDATES:
        try:
            t0 = time.time()
            outs = run_llm(name, todo)
            bad = np.mean([len(o.split()) < 30 for o in outs.values()])
            print(f'\n{name}: {time.time()-t0:.0f}s, bad outputs {bad:.0%}')
            if bad > 0.2: raise RuntimeError('too many bad outputs')
            EXP.update(outs); json.dump(EXP, open(cache_path, 'w'))
            break
        except Exception as e:
            print('\n❌', name, '→', e); gc.collect(); torch.cuda.empty_cache()

exp_train = [EXP.get(t, '') for t in train_q['short']]
exp_test  = [EXP.get(t, '') for t in test_q['query']]
print('Expansions ready:', sum(bool(e) for e in exp_train + exp_test), '/', len(exp_train + exp_test))
print('\nEXAMPLE:', test_q['query'][0], '\n→', exp_test[0][:800])

In [ ]:
# 6) قنوات البحث بالكلمات
CH = {'train': {}, 'test': {}}
CH['train']['bm25_q'] = norm(bm25.scores(train_q['short']))
CH['test']['bm25_q']  = norm(bm25.scores(test_q['query']))
if any(exp_train):
    CH['train']['bm25_e'] = norm(bm25.scores([q + '\n' + e for q, e in zip(train_q['short'], exp_train)]))
    CH['test']['bm25_e']  = norm(bm25.scores([q + '\n' + e for q, e in zip(test_q['query'], exp_test)]))
for k, S in CH['train'].items(): print(k, round(score(S), 2))

## 7) البحث الدلالي (Embeddings)
ترميز الـ 49 ألف مقطع ياخذ تقريباً 5–10 دقايق، ويتحفظ عشان لو أعدتي التشغيل يكون فوري.

In [ ]:
USE_DENSE = HAS_GPU
EMB_CANDIDATES = [
    ('Qwen/Qwen3-Embedding-0.6B', 'Instruct: Given a biology question, retrieve passages that explain the scientific concepts needed to answer it\nQuery: '),
    ('BAAI/bge-large-en-v1.5', 'Represent this sentence for searching relevant passages: '),
]
if USE_DENSE:
    from sentence_transformers import SentenceTransformer
    for name, qpref in EMB_CANDIDATES:
        try:
            t0 = time.time()
            emb = SentenceTransformer(name, device='cuda', model_kwargs={'torch_dtype': torch.float16})
            emb.max_seq_length = 512
            path = f"{OUT}/doc_emb_{name.split('/')[-1]}.npy"
            if os.path.exists(path):
                D = np.load(path)
            else:
                D = emb.encode(corpus.text.tolist(), batch_size=32, normalize_embeddings=True,
                               show_progress_bar=True, convert_to_numpy=True).astype(np.float16)
                np.save(path, D)
            enc = lambda xs: emb.encode([qpref + x for x in xs], batch_size=16, normalize_embeddings=True,
                                        convert_to_numpy=True).astype(np.float32)
            Df = D.astype(np.float32)
            CH['train']['dense_q'] = norm(enc(list(train_q['short'])) @ Df.T)
            CH['test']['dense_q']  = norm(enc(list(test_q['query'])) @ Df.T)
            if any(exp_train):
                CH['train']['dense_e'] = norm(enc([q + '\n' + e for q, e in zip(train_q['short'], exp_train)]) @ Df.T)
                CH['test']['dense_e']  = norm(enc([q + '\n' + e for q, e in zip(test_q['query'], exp_test)]) @ Df.T)
            del emb, Df; gc.collect(); torch.cuda.empty_cache()
            print(f'{name} done in {time.time()-t0:.0f}s'); break
        except Exception as e:
            print('❌', name, '→', e); gc.collect(); torch.cuda.empty_cache()
for k, S in CH['train'].items(): print(k, round(score(S), 2))

In [ ]:
# 8) نختار أفضل أوزان للدمج تلقائياً على أسئلة التدريب (القصيرة)
names = list(CH['train'])
results = []
for ws in itertools.product([0, 0.5, 1, 2], repeat=len(names)):
    if sum(ws) == 0 or max(ws) != 2 and max(ws) != 1: continue
    F = sum(w * CH['train'][n] for w, n in zip(ws, names) if w)
    for beta in [0, 0.15, 0.3]:
        results.append((score(smooth(F, beta)), ws, beta))
results.sort(key=lambda r: -r[0])
print('channels:', names)
for r in results[:8]: print(round(r[0], 2), r[1], 'beta', r[2])
BEST_W, BEST_BETA = results[0][1], results[0][2]

def fuse(split, ws=BEST_W, beta=BEST_BETA):
    return smooth(sum(w * CH[split][n] for w, n in zip(ws, names) if w), beta)
F_train, F_test = fuse('train'), fuse('test')

# بديل "آمن" للتسليم: كل القنوات بوزن متساوٍ
EQ_W = tuple(1 for _ in names)
print('Equal weights:', round(score(fuse('train', EQ_W, 0)), 2))

## 9) إعادة الترتيب (Reranker) — اختياري
يعيد ترتيب أفضل 100 مقطع. الكود يقرر لحاله: لو ما حسّن النتيجة على التدريب، وزنه يصير صفر.

In [ ]:
USE_RERANK = HAS_GPU
K = 100
cand_train, cand_test = topk(F_train, K), topk(F_test, K)
RR_W = 0
if USE_RERANK:
    try:
        from sentence_transformers import CrossEncoder
        t0 = time.time()
        ce = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512, device='cuda',
                          model_kwargs={'torch_dtype': torch.float16})
        def rr(queries, cands):
            pairs = [(q, corpus.text.values[i]) for q, c in zip(queries, cands) for i in c]
            return np.asarray(ce.predict(pairs, batch_size=32, show_progress_bar=True)).reshape(len(queries), -1)
        R_train, R_test = rr(list(train_q['short']), cand_train), rr(list(test_q['query']), cand_test)
        del ce; gc.collect(); torch.cuda.empty_cache()
        print(f'rerank done in {time.time()-t0:.0f}s')

        def mm(A): return (A - A.min(1, keepdims=True)) / (A.max(1, keepdims=True) - A.min(1, keepdims=True) + 1e-9)
        def combine(F, cand, R, w):
            base = mm(np.take_along_axis(F, cand, 1))
            fin = base + w * mm(R)
            return np.take_along_axis(cand, np.argsort(-fin, 1), 1)
        def score_rank(T):
            return 100 * np.mean([ndcg10(list(doc_ids[t]), gold[q]) for t, q in zip(T, TRAIN_IDS)])
        for w in [0, 0.25, 0.5, 1, 2, 4]:
            s = score_rank(combine(F_train, cand_train, R_train, w))
            print('rerank weight', w, '→', round(s, 2))
            if w == 0 or s > best_rr: best_rr, RR_W = s, w
    except Exception as e:
        print('❌ rerank →', e)
print('Chosen rerank weight:', RR_W)

In [ ]:
# 10) ملفات التسليم
def write(rank_matrix, fname):
    sub = pd.DataFrame({'query_id': test_q.query_id,
                        'doc_ids': [' '.join(doc_ids[t][:10]) for t in rank_matrix]})
    assert set(sub.query_id) == set(sample.query_id)
    sub.to_csv(f'{OUT}/{fname}', index=False)
    print('saved', fname)

if RR_W > 0:
    write(combine(F_test, cand_test, R_test, RR_W), 'submission.csv')
else:
    write(topk(F_test, 10), 'submission.csv')
write(topk(F_test, 10), 'submission_no_rerank.csv')
write(topk(fuse('test', EQ_W, 0), 10), 'submission_equal_weights.csv')
pd.read_csv(f'{OUT}/submission.csv').head()

In [ ]:
for ws in [(1,1,1,1), (0,1,0,1), (0.5,1,0.5,1), (0,2,0,1)]:
    print(ws, round(score(fuse('train', ws, 0)), 2))
write(topk(fuse('test', (0,1,0,1), 0), 10), 'submission_exp_only.csv')